# 01 — Setup, model loading, and a first feature-extraction tour

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

**This notebook walks through the smallest possible end-to-end loop:**

1. Install dependencies and pick a device (Colab T4/A100 or local).
2. Load a small Transformer (`gpt2`, 124 M) via [TransformerLens](https://transformerlensorg.github.io/TransformerLens/).
3. Tokenize a single prompt and run a forward pass with caching.
4. Extract residual-stream activations per layer × token into a NumPy array — the *cumulative* representation built up across the network.
5. Pull attention patterns for one layer; visualise every head as a small heatmap grid.
6. Highlight which previous token each head attends to from the final position (a quick interpretability fingerprint).
7. Save residuals and attentions to `data/processed/` for downstream notebooks.

**Runtime.** ~30 s end-to-end on free Colab T4; ~1–2 min on CPU.

**Why GPT-2 small?** Zero auth, downloads in seconds, runs anywhere. We will move to Pythia-1.4B / Llama-3.1-8B in notebook 02 once the workflow is validated.

## 0. Colab / local setup

Installs the only dependency that does not ship with Colab by default. If running locally inside the project venv (`pip install -e .`), the install is a no-op.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Lightweight install — only what this notebook needs.
    !pip install -q transformer-lens

    # HF token from Colab Secrets (only needed for gated models like Llama-3).
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass

    # Persistent state — uncomment to write outputs to Drive.
    # from google.colab import drive; drive.mount("/content/drive")
    # WORK_DIR = "/content/drive/MyDrive/EmoVecLLM"
    WORK_DIR = "/content/EmoVecLLM"
else:
    WORK_DIR = os.path.expanduser("~/Projects/EmoVecLLM")

os.makedirs(os.path.join(WORK_DIR, "data", "processed"), exist_ok=True)
print(f"WORK_DIR = {WORK_DIR}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

## 1. Load a model

TransformerLens's `HookedTransformer` is a thin wrapper that exposes named hooks at every component (residual stream, attention pattern, MLP, etc.). We will use those hooks directly to pull activations and, in later notebooks, to inject steering vectors.

In [ ]:
MODEL_NAME = "gpt2"   # 124 M params. Swap for "EleutherAI/pythia-1.4b" once validated.

model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
d_model = model.cfg.d_model
print(f"{MODEL_NAME}: {n_layers} layers × {n_heads} heads, d_model = {d_model}")

## 2. Pick a prompt and tokenize

We use a short narrative sentence with mild affective content. The prompt is small enough that the attention heatmaps are legible; large enough that there is real structure to look at. Tokens are byte-pair, so leading whitespace becomes part of the token string (`' heard'` ≠ `'heard'`).

In [ ]:
prompt = "She heard the door creak open and her heart began to race"

token_strs = model.to_str_tokens(prompt)
n_tokens = len(token_strs)
print(f"{n_tokens} tokens:")
for i, t in enumerate(token_strs):
    print(f"  {i:2d}  {t!r}")

## 3. Forward pass with cache → per-layer residual stream

`run_with_cache` runs a forward pass and returns every named activation. The two we want here:

- `blocks.{i}.hook_resid_post` — residual stream **after** block *i* (shape `(seq_len, d_model)` once we strip the batch dim).
- `blocks.{i}.attn.hook_pattern` — softmaxed attention weights for block *i* (shape `(n_heads, seq_q, seq_k)`).

We stack residuals into a single NumPy array `resid` of shape `(n_layers, seq_len, d_model)`. Each row `resid[i]` *is* the cumulative representation of the prompt after layer *i* — the network builds the meaning of each token incrementally as we move up the stack.

In [ ]:
with torch.no_grad():
    _, cache = model.run_with_cache(prompt, remove_batch_dim=True)

resid = np.stack(
    [cache[f"blocks.{i}.hook_resid_post"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)
embed = cache["hook_embed"].cpu().float().numpy()

print(f"resid.shape = {resid.shape}   # (n_layers, seq_len, d_model)")
print(f"embed.shape = {embed.shape}   # (seq_len, d_model)")

## 4. Cumulative view — residual-stream norm per layer × token

A single heatmap of `‖resid[i, t]‖₂` over layers (y) and tokens (x). The norm climbs through the network as each block writes more information into the residual stream. Spikes flag tokens that the network is putting unusual effort into representing.

In [ ]:
resid_norm = np.linalg.norm(resid, axis=-1)   # (n_layers, seq_len)

fig, ax = plt.subplots(figsize=(0.5 * n_tokens + 2, 0.3 * n_layers + 1.5))
im = ax.imshow(resid_norm, aspect="auto", cmap="magma", origin="lower")
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(token_strs, rotation=45, ha="right")
ax.set_yticks(range(n_layers))
ax.set_ylabel("layer")
ax.set_title("‖residual stream‖₂  per layer × token")
fig.colorbar(im, ax=ax, shrink=0.8, label="L2 norm")
fig.tight_layout()
plt.show()

## 5. Attention patterns for one layer — head grid

Each subplot is the `(seq_q, seq_k)` attention matrix for one head, in a single chosen layer. Mid-stack layers tend to show the richest mix (induction-style off-diagonals, previous-token heads, etc.). Lower triangular by causal masking.

In [ ]:
LAYER = n_layers // 2

attn = cache[f"blocks.{LAYER}.attn.hook_pattern"].cpu().float().numpy()   # (heads, q, k)
print(f"attn.shape = {attn.shape}")

ncols = 4
nrows = int(np.ceil(n_heads / ncols))
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.5 * ncols, 2.3 * nrows),
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

for h in range(n_heads):
    ax = axes.flat[h]
    ax.imshow(attn[h], cmap="Blues", vmin=0, vmax=1, aspect="equal")
    ax.set_title(f"L{LAYER} H{h}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
for h in range(n_heads, nrows * ncols):
    axes.flat[h].axis("off")

fig.suptitle(f"Attention patterns — layer {LAYER}", y=1.02)
plt.show()

## 6. Highlight — where does the *final* token look?

For each head in the same layer, the row of attention weights from the last query position. Two views:

- A `(heads × keys)` heatmap with token strings on the x-axis — easy to spot "previous-token heads" (bright on the diagonal-1) and "BOS heads" (bright on token 0).
- A horizontal bar of per-head **attention entropy** (in nats). Low entropy = head concentrates on one or two tokens; high entropy = head spreads attention diffusely.

In [ ]:
final_q = n_tokens - 1
attn_from_last = attn[:, final_q, :]   # (n_heads, seq_len)

eps = 1e-12
entropy = -(attn_from_last * np.log(attn_from_last + eps)).sum(axis=-1)

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(0.4 * n_tokens + 6, 0.25 * n_heads + 1.8),
    gridspec_kw={"width_ratios": [3, 1]},
    constrained_layout=True,
)
im = ax1.imshow(attn_from_last, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax1.set_xticks(range(n_tokens))
ax1.set_xticklabels(token_strs, rotation=45, ha="right")
ax1.set_yticks(range(n_heads))
ax1.set_ylabel("head")
ax1.set_title(f"Attention from final token  —  layer {LAYER}")
fig.colorbar(im, ax=ax1, shrink=0.8, label="weight")

ax2.barh(range(n_heads), entropy, color="#4477aa")
ax2.set_yticks(range(n_heads))
ax2.invert_yaxis()
ax2.set_xlabel("entropy (nats)")
ax2.set_title("per-head\nentropy")
plt.show()

print(f"\nMost-attended-to token from final query (layer {LAYER}):")
for h in range(n_heads):
    k = int(attn_from_last[h].argmax())
    w = float(attn_from_last[h, k])
    print(f"  head {h:2d}  →  token {k:2d}  {token_strs[k]!r:>14}   (w={w:.2f})")

## 7. Save features to disk

Stack attentions across all layers and save residuals + attentions + tokens as a single `.npz`. Notebook 02 will load this file and start building the actual emotion-vector pipeline on top.

In [ ]:
attn_all = np.stack(
    [cache[f"blocks.{i}.attn.hook_pattern"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)   # (n_layers, n_heads, seq_q, seq_k)

out = os.path.join(WORK_DIR, "data", "processed", "nb01_features.npz")
np.savez(
    out,
    resid=resid,
    embed=embed,
    attn=attn_all,
    tokens=np.array(token_strs, dtype=object),
    model_name=MODEL_NAME,
)
print(f"Saved → {out}")
print(f"  resid: {resid.shape}")
print(f"  embed: {embed.shape}")
print(f"  attn:  {attn_all.shape}")

---

## Next steps

- **Notebook 02 (`02_emotion_word_list_and_prompts.ipynb`)** — source the 171-emotion list and build paired emotion / neutral story prompts (Stage 1 of `meta/TODO.md`).
- **Adapter wrap** — promote the `from_pretrained` + `run_with_cache` pattern used here into `src/emovecllm/models.TransformerLensAdapter` (Stage 2).
- **Try a bigger model** — re-run with `MODEL_NAME = "EleutherAI/pythia-1.4b"` once you confirm the workflow on `gpt2`. On free Colab T4 this still fits comfortably in fp16.